# CHESCA Commitment Mesh Ablation

이 notebook은 기존 결과와 notebook을 덮어쓰지 않는 새 실행 파일입니다. 기존 `chesca_mesh`를 기준으로, commitment ledger, local recovery, soft throughput budget이 각각 성능을 어떻게 바꾸는지 ablation합니다.

## 1. Google Drive 연결

`chesca_vs_mesh` 폴더 전체를 `MyDrive` 바로 아래에 올린 뒤 실행합니다.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/chesca_vs_mesh')
OFFICIAL_DIR = PROJECT_DIR / 'CHESCA-main'
assert (OFFICIAL_DIR / 'checa' / 'agent.py').exists(), f'공식 CHESCA 폴더를 찾을 수 없습니다: {OFFICIAL_DIR}'
assert (PROJECT_DIR / 'src' / 'chesca_vs_mesh' / 'commitment_mesh_agent.py').exists(), 'commitment mesh 코드가 누락되었습니다.'
print('Project:', PROJECT_DIR)
print('Official source:', OFFICIAL_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/chesca_vs_mesh
Official source: /content/drive/MyDrive/chesca_vs_mesh/CHESCA-main


## 2. Colab 설치

CityLearn 2.1b12는 프로젝트의 `third_party`에 포함된 런타임을 사용합니다. 오래된 PyPI CityLearn 의존성이 Colab의 NumPy/Pandas와 충돌하지 않도록 CityLearn 자체는 pip로 설치하지 않습니다.

In [ ]:
%pip install -q "gym==0.26.2" "simplejson>=3.19" "xgboost>=1.7,<3"

import sys
VENDORED_CITYLEARN = PROJECT_DIR / 'third_party' / 'CityLearn-2.1b12'
assert (VENDORED_CITYLEARN / 'citylearn' / 'citylearn.py').exists(), f'CityLearn runtime을 찾을 수 없습니다: {VENDORED_CITYLEARN}'
sys.path.insert(0, str(VENDORED_CITYLEARN))

import numpy as np
import pandas as pd
import scipy
import torch
import xgboost
import citylearn

print('numpy:', np.__version__, 'pandas:', pd.__version__, 'scipy:', scipy.__version__)
print('torch:', torch.__version__, 'xgboost:', xgboost.__version__)
print('CityLearn:', citylearn.__version__, citylearn.__file__)
assert citylearn.__version__ == '2.1b12'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 20.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
numpy: 2.0.2 pandas: 2.2.2 scipy: 1.16.3
torch: 2.11.0+cu128 xgboost: 2.1.4
CityLearn: 2.1b12 /content/drive/MyDrive/chesca_vs_mesh/third_party/CityLearn-2.1b12/citylearn/__init__.py


## 3. 프로젝트 로드와 설정

Ablation controller는 다음 의미입니다.

- `chesca_commitment_ledger_mesh`: 방전 debt 기록 + debt가 큰 peer의 추가 방전을 약간 불리하게 평가
- `chesca_commitment_recovery_mesh`: ledger + 낮은 district stress에서 local recovery charge 우선
- `chesca_commitment_budget_mesh`: debt 없이 rolling throughput soft friction만 적용
- `chesca_commitment_full_mesh`: ledger + recovery + budget 전체 적용

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from chesca_vs_mesh import MeshConfig, available_datasets
from chesca_vs_mesh.commitment_mesh_agent import CommitmentConfig
from chesca_vs_mesh.commitment_evaluation import CommitmentBenchmarkSuite

print('Available bundled datasets:')
print(available_datasets())

DATASET = 'citylearn_challenge_2023_phase_3_1'
EPISODE_STEPS = None  # None: official schema 전체 기간. 빠른 연결 확인은 71 등으로 변경.
TAG = 'commitment_ablation_v1'
ABLATION_CONTROLLERS = [
    'chesca_official',
    'chesca_mesh',
    'chesca_commitment_ledger_mesh',
    'chesca_commitment_recovery_mesh',
    'chesca_commitment_budget_mesh',
    'chesca_commitment_full_mesh',
]

previous_mesh_config = MeshConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
)
commitment_config = CommitmentConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
    debt_weight=0.18,
    recovery_weight=0.42,
    budget_weight=0.12,
    max_debt_soc=0.18,
    soft_budget_soc=0.22,
)
suite = CommitmentBenchmarkSuite(
    output_directory=PROJECT_DIR / 'results',
    mesh_config=previous_mesh_config,
    commitment_config=commitment_config,
    power_outage_seed=None,
)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Available bundled datasets:
['citylearn_challenge_2023_phase_1', 'citylearn_challenge_2023_phase_2_local_evaluation', 'citylearn_challenge_2023_phase_2_online_evaluation_1', 'citylearn_challenge_2023_phase_2_online_evaluation_2', 'citylearn_challenge_2023_phase_2_online_evaluation_3', 'citylearn_challenge_2023_phase_3_1', 'citylearn_challenge_2023_phase_3_2', 'citylearn_challenge_2023_phase_3_3', 'warm_up']


## 4. Commitment ablation 실행

이 셀은 단일 private schema에서 6개 controller를 비교합니다. 기존 결과를 보존하기 위해 새 태그 `commitment_ablation_v1`에 저장합니다.

In [ ]:
result = suite.compare_controllers(
    dataset_name=DATASET,
    controllers=ABLATION_CONTROLLERS,
    episode_steps=EPISODE_STEPS,
    tag=TAG,
)

score_columns = [
    'controller', 'challenge_cost', 'comfort_cost', 'emissions_cost',
    'grid_cost', 'resilience_cost', 'challenge_cost_change_vs_chesca_pct',
    'grid_cost_change_vs_chesca_pct', 'resilience_cost_change_vs_chesca_pct',
]
display(result.summary[score_columns])
display(result.citylearn_metrics)
print('Results saved to:', result.output_directory)

,controller,challenge_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,challenge_cost_change_vs_chesca_pct,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct
0,chesca_official,0.590169,0.130315,0.925583,0.957768,0.570619,0.000000,0.000000,0.000000
1,chesca_mesh,0.589098,0.130710,0.925827,0.943553,0.580788,-0.181399,-1.484100,1.782102
2,chesca_commitment_ledger_mesh,0.586918,0.130391,0.925876,0.945475,0.571902,-0.550852,-1.283513,0.224792
3,chesca_commitment_recovery_mesh,0.588135,0.130621,0.927187,0.946242,0.574523,-0.344679,-1.203343,0.684158
4,chesca_commitment_budget_mesh,0.589030,0.130573,0.926056,0.946062,0.578112,-0.193013,-1.222183,1.313085
5,chesca_commitment_full_mesh,0.586544,0.130391,0.926412,0.943842,0.572110,-0.614197,-1.454016,0.261284


,controller,carbon_emissions_total,discomfort_proportion,ramping_average,daily_one_minus_load_factor_average,daily_peak_average,annual_peak_average,one_minus_thermal_resilience_proportion,power_outage_normalized_unserved_energy_total,average_score
0,chesca_official,0.925583,0.130315,0.850411,0.958645,0.889024,1.132991,0.788781,0.352457,0.590169
1,chesca_mesh,0.925827,0.130710,0.819864,0.960908,0.891153,1.102288,0.805797,0.355779,0.589098
2,chesca_commitment_ledger_mesh,0.925876,0.130391,0.825045,0.955415,0.886306,1.115132,0.792749,0.351054,0.586918
3,chesca_commitment_recovery_mesh,0.927187,0.130621,0.839194,0.956157,0.888522,1.101097,0.800685,0.348360,0.588135
4,chesca_commitment_budget_mesh,0.926056,0.130573,0.834603,0.959122,0.889475,1.101049,0.805050,0.351173,0.589030
5,chesca_commitment_full_mesh,0.926412,0.130391,0.830527,0.956676,0.887802,1.100362,0.792749,0.351471,0.586544


Results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/citylearn_challenge_2023_phase_3_1/commitment_ablation_v1


## 5. Commitment 진단

아래 표는 성능이 왜 좋아지거나 나빠졌는지를 보기 위한 진단입니다. `extra_discharge_soc_total`은 mesh가 CHESCA 대비 추가로 방전한 SOC, `extra_charge_soc_total`은 추가 충전한 SOC입니다. `debt_created_soc_total`과 `debt_repaid_soc_total`의 차이가 크면 회복이 제대로 작동하지 않는 것입니다.

In [ ]:
diagnostic_columns = [
    'controller', 'message_count', 'changed_trade_steps', 'changed_trade_step_pct',
    'extra_discharge_soc_total', 'extra_charge_soc_total',
    'debt_created_soc_total', 'debt_repaid_soc_total',
    'mean_total_debt_soc', 'max_total_debt_soc', 'max_peer_debt_soc',
    'relief_action_steps', 'recovery_action_steps', 'mean_budget_use_soc',
]
available_columns = [c for c in diagnostic_columns if c in result.summary.columns]
display(result.summary[available_columns])

commitment_negotiations = result.negotiations[
    result.negotiations['controller'].str.contains('commitment', na=False)
].copy()
if commitment_negotiations.empty:
    print('Commitment negotiation log가 없습니다.')
else:
    display(commitment_negotiations.groupby('controller')[[
        'changed_peers', 'relief_selected_peers', 'recovery_selected_peers',
        'debt_created_soc', 'debt_repaid_soc', 'total_debt_soc',
        'mean_budget_use_soc', 'predicted_grid_delta',
    ]].mean())
    display(commitment_negotiations.tail(30))

,controller,message_count,changed_trade_steps,changed_trade_step_pct,extra_discharge_soc_total,extra_charge_soc_total,debt_created_soc_total,debt_repaid_soc_total,mean_total_debt_soc,max_total_debt_soc,max_peer_debt_soc,relief_action_steps,recovery_action_steps,mean_budget_use_soc
0,chesca_official,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chesca_mesh,194670,1774,82.015719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chesca_commitment_ledger_mesh,194670,719,33.240869,1.08,123.46,1.08,0.00,0.822931,1.08,0.20,8.0,712.0,0.229584
3,chesca_commitment_recovery_mesh,194670,1134,52.427184,64.68,152.94,64.68,64.54,0.161729,0.72,0.12,337.0,801.0,0.400524
4,chesca_commitment_budget_mesh,194670,964,44.567730,95.54,26.52,0.00,0.00,0.000000,0.00,0.00,697.0,279.0,0.223937
5,chesca_commitment_full_mesh,194670,1053,48.682386,42.12,72.10,42.12,41.92,0.111586,0.58,0.12,443.0,633.0,0.209940


,changed_peers,relief_selected_peers,recovery_selected_peers,debt_created_soc,debt_repaid_soc,total_debt_soc,mean_budget_use_soc,predicted_grid_delta
controller,,,,,,,,
chesca_commitment_budget_mesh,2.017106,1.527508,0.489598,0.000000,0.000000,0.000000,0.223937,-0.122193
chesca_commitment_full_mesh,1.853444,0.688396,1.165049,0.019473,0.019380,0.111586,0.209940,0.037555
chesca_commitment_ledger_mesh,1.822006,0.012483,1.809524,0.000499,0.000000,0.822931,0.229584,0.179372
chesca_commitment_recovery_mesh,2.852057,0.749884,2.102173,0.029903,0.029838,0.161729,0.400524,0.112434


,controller,step,hour,active_peers,changed_peers,official_predicted_grid,negotiated_predicted_grid,predicted_grid_delta,district_target,final_shadow_signal,logical_message_count,relief_selected_peers,recovery_selected_peers,debt_created_soc,debt_repaid_soc,extra_discharge_soc,extra_charge_soc,total_debt_soc,max_debt_soc,mean_budget_use_soc
10785,chesca_commitment_full_mesh,2177,18,6,3,12.257563,12.059563,-1.980000e-01,9.258351,1.361701,90,3.0,0.0,0.06,0.000000e+00,0.06,0.00,2.600000e-01,6.000000e-02,0.223333
10786,chesca_commitment_full_mesh,2178,19,6,5,15.058204,14.526204,-5.320000e-01,9.258351,1.817318,90,5.0,0.0,0.14,0.000000e+00,0.14,0.00,4.000000e-01,8.000000e-02,0.213333
10787,chesca_commitment_full_mesh,2179,20,6,0,14.880567,14.880567,0.000000e+00,9.258351,1.583204,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.210000
10788,chesca_commitment_full_mesh,2180,21,6,0,11.763681,11.763681,0.000000e+00,9.258351,0.638666,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.170000
10789,chesca_commitment_full_mesh,2181,22,6,0,10.525310,10.525310,0.000000e+00,9.258351,0.351949,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.170000
10790,chesca_commitment_full_mesh,2182,23,6,0,10.423525,10.423525,1.776357e-15,9.298928,0.308828,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.170000
10791,chesca_commitment_full_mesh,2183,24,6,0,11.635987,11.635987,0.000000e+00,9.298928,0.781032,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.170000
10792,chesca_commitment_full_mesh,2184,1,6,0,11.164675,11.164675,-1.776357e-15,9.298928,0.457632,90,0.0,0.0,0.00,0.000000e+00,0.00,0.00,4.000000e-01,8.000000e-02,0.170000
10793,chesca_commitment_full_mesh,2185,2,6,6,9.874620,10.750620,8.760000e-01,9.298928,0.009628,90,0.0,6.0,0.00,2.400000e-01,0.00,0.24,1.600000e-01,4.000000e-02,0.200000
10794,chesca_commitment_full_mesh,2186,3,6,6,7.663862,8.407862,7.440000e-01,9.342092,-0.185407,90,0.0,6.0,0.00,1.600000e-01,0.00,0.20,1.040834e-17,6.938894e-18,0.233333


## 6. 논문형 Public Cost / Private Cost

전체 ablation을 public/private 6개 schema에 모두 돌리면 시간이 오래 걸립니다. 기본값은 `chesca_official`, 기존 `chesca_mesh`, `chesca_commitment_full_mesh`만 비교합니다. 모든 ablation을 평가하려면 `PUBLIC_PRIVATE_CONTROLLERS = ABLATION_CONTROLLERS`로 바꾸면 됩니다.

In [ ]:
RUN_PUBLIC_PRIVATE = True
PUBLIC_PRIVATE_CONTROLLERS = [
    'chesca_official',
    'chesca_mesh',
    'chesca_commitment_full_mesh',
]
# PUBLIC_PRIVATE_CONTROLLERS = ABLATION_CONTROLLERS  # 모든 ablation을 public/private로 평가할 때 사용

if RUN_PUBLIC_PRIVATE:
    leaderboard = suite.compare_public_private_costs(
        controllers=PUBLIC_PRIVATE_CONTROLLERS,
        episode_steps=None,
        tag='paper_public_private_commitment_v1',
    )
    display(leaderboard.paper_table)
    display(leaderboard.summary)

    diagnostic_columns = [
        'split', 'run_id', 'dataset', 'controller', 'challenge_cost',
        'grid_cost', 'resilience_cost',
        'grid_cost_change_vs_chesca_pct',
        'resilience_cost_change_vs_chesca_pct',
    ]
    display(leaderboard.runs[diagnostic_columns])
    print('Public/private results saved to:', leaderboard.output_directory)
else:
    print('RUN_PUBLIC_PRIVATE=True로 변경하면 Public/Private Cost 비교를 실행합니다.')

,controller,Private Cost,Public Cost,Private Cost Change vs CHESCA (%),Public Cost Change vs CHESCA (%)
0,chesca_commitment_full_mesh,0.563306,0.502552,-0.779087,-1.151755
1,chesca_mesh,0.566371,0.502961,-0.239082,-1.071209
2,chesca_official,0.567729,0.508408,0.000000,0.000000


,split,controller,leaderboard_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,leaderboard_cost_change_vs_chesca_pct
0,private,chesca_commitment_full_mesh,0.563306,0.128876,0.929242,0.920219,0.518843,-0.779087
1,private,chesca_mesh,0.566371,0.129195,0.928669,0.920573,0.528580,-0.239082
2,private,chesca_official,0.567729,0.128942,0.928365,0.934023,0.520009,0.000000
3,public,chesca_commitment_full_mesh,0.502552,0.070437,0.952625,0.873233,0.413962,-1.151755
4,public,chesca_mesh,0.502961,0.070449,0.951868,0.879749,0.409050,-1.071209
5,public,chesca_official,0.508408,0.070520,0.951344,0.888128,0.418929,0.000000


,split,run_id,dataset,controller,challenge_cost,grid_cost,resilience_cost,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct
0,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.536594,0.885566,0.513373,0.000000,0.000000
1,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.533249,0.874186,0.513388,-1.285026,0.002899
2,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_commitment_full_mesh,0.532863,0.872668,0.513371,-1.456485,-0.000414
3,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.520862,0.875704,0.471915,0.000000,0.000000
4,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.518596,0.868274,0.471619,-0.848515,-0.062703
5,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_commitment_full_mesh,0.518268,0.867296,0.471288,-0.960181,-0.132826
6,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.467766,0.903115,0.271499,0.000000,0.000000
7,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.457040,0.896788,0.242143,-0.700484,-10.812737
8,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_commitment_full_mesh,0.456525,0.879736,0.257225,-2.588671,-5.257447
9,private,1,citylearn_challenge_2023_phase_3_1,chesca_official,0.590169,0.957768,0.570619,0.000000,0.000000


Public/private results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/paper_public_private_commitment_v1
